# Prepared radiation calculations

Use a caller-owned preparation cache for independent calculations of one explicit
illustrative physical Building. Compare the retained v1 policy with the explicitly
selected v2 midpoint/angular approximation. These are software examples, not
measured-building validation.

Install the RC platform and notebook extras with the [EPSM supported combination](https://github.com/SB-Chalmers/epsm/blob/main/compatibility.json). The
packaged Goteborg EPW retains its original source headers. Outputs stay in memory;
no simulation directory is created. Ground coupling and omitted supply controls
are explicitly selected approximations. Geometric shading and dynamic sky
longwave exchange are unsupported.

In [ ]:
from importlib.resources import files

import numpy as np
from epsm_core import Building, Weather

from epsm_rc import PreparationCache, RCSettings, simulate_building

building = Building(preset="illustrative-reference")
weather = Weather.from_epw(files("epsm_rc").joinpath("data/epw/goteborg.epw"))
cache = PreparationCache(max_entries=32)
legacy = RCSettings.from_policy(
    "epsm_rc_5r1c_v1",
    airflow_policy="sensible_only_unlimited_supply",
    ground_policy="conductance_multiplier",
)
angular = RCSettings.from_policy(
    "epsm_rc_5r1c_v2",
    airflow_policy="sensible_only_unlimited_supply",
    ground_policy="conductance_multiplier",
)

## Reuse preparation, keep independent thermal states

Cache use must preserve every hourly value and the calculation identity. Each call
still verifies the weather and implementation contents and creates new thermal
state. This cache is distinct from durable study-result caching.

In [ ]:
uncached = simulate_building(building, weather=weather, settings=legacy)
prepared = simulate_building(building, weather=weather, settings=legacy, preparation_cache=cache)
repeated = simulate_building(building, weather=weather, settings=legacy, preparation_cache=cache)
assert uncached.run_id == prepared.run_id == repeated.run_id
for name, values in uncached.hourly.items():
    np.testing.assert_array_equal(values, prepared.hourly[name])
    np.testing.assert_array_equal(values, repeated.hourly[name])
assert cache.hits > 0
print(cache)

## Inspect the explicitly different radiation model

V2 uses midpoint solar position and separate beam/diffuse optics. Window heat gain
includes transmitted solar and nominal inward absorbed solar. It is not directly
comparable to transmitted solar alone. The angular model does not solve the
complete EnergyPlus window heat balance. The previous fixed longwave approximation
is retained; changes in annual heating are not proof of prediction accuracy.

In [ ]:
improved = simulate_building(building, weather=weather, settings=angular, preparation_cache=cache)
assert improved.run_id != prepared.run_id
np.testing.assert_array_equal(improved.time_axis, prepared.time_axis)
assert improved.diagnostics["accepted"]
np.testing.assert_allclose(
    improved.hourly["window_solar_heat_gain_w"],
    improved.hourly["window_transmitted_solar_w"]
    + improved.hourly["window_inward_absorbed_solar_w"],
    rtol=2e-15,
    atol=2e-11,
)
print(
    {
        "legacy_heating_kwh": prepared.scalars["space_heating_demand_kwh"],
        "angular_heating_kwh": improved.scalars["space_heating_demand_kwh"],
    }
)
print(improved.diagnostics["radiation"]["window_optics_method"])
cache.clear()
assert len(cache) == 0